In [2]:
import pandas as pd
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
print("Train shape:", train.shape)
print("Test shape:", test.shape)
train.head()

Train shape: (8693, 14)
Test shape: (4277, 13)


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [3]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   str    
 1   HomePlanet    8492 non-null   str    
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   str    
 4   Destination   8511 non-null   str    
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   str    
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(2), str(5)
memory usage: 891.5+ KB


In [4]:
missing = train.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(train)) * 100
pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct.round(2)})

,Missing Count,Missing %
CryoSleep,217,2.50
ShoppingMall,208,2.39
VIP,203,2.34
HomePlanet,201,2.31
Name,200,2.30
Cabin,199,2.29
VRDeck,188,2.16
Spa,183,2.11
FoodCourt,183,2.11
Destination,182,2.09


In [5]:
train.groupby('CryoSleep')['Transported'].mean()

CryoSleep
False    0.328921
True     0.817583
Name: Transported, dtype: float64

In [6]:
train.groupby('CryoSleep')[['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']].mean()

,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
CryoSleep,,,,,
False,350.146772,713.004316,270.586504,486.09294,475.716165
True,0.000000,0.000000,0.000000,0.00000,0.000000


In [7]:
train['Cabin'].head(10)

0    B/0/P
1    F/0/S
2    A/0/S
3    A/0/S
4    F/1/S
5    F/0/P
6    F/2/S
7    G/0/S
8    F/3/S
9    B/1/P
Name: Cabin, dtype: str

In [8]:
train[['Deck', 'CabinNum', 'Side']] = train['Cabin'].str.split('/', expand=True)
test[['Deck', 'CabinNum', 'Side']] = test['Cabin'].str.split('/', expand=True)
train[['Cabin', 'Deck', 'CabinNum', 'Side']].head()

,Cabin,Deck,CabinNum,Side
0,B/0/P,B,0,P
1,F/0/S,F,0,S
2,A/0/S,A,0,S
3,A/0/S,A,0,S
4,F/1/S,F,1,S


In [9]:
print(train.groupby('Deck')['Transported'].mean().sort_values(ascending=False))
print()
print(train.groupby('Side')['Transported'].mean())

Deck
B    0.734275
C    0.680054
G    0.516217
A    0.496094
F    0.439871
D    0.433054
E    0.357306
T    0.200000
Name: Transported, dtype: float64

Side
P    0.451260
S    0.555037
Name: Transported, dtype: float64


In [10]:
print(train.groupby('HomePlanet')['Transported'].mean().sort_values(ascending=False))
print()
print(train.groupby('Destination')['Transported'].mean().sort_values(ascending=False))

HomePlanet
Europa    0.658846
Mars      0.523024
Earth     0.423946
Name: Transported, dtype: float64

Destination
55 Cancri e      0.610000
PSO J318.5-22    0.503769
TRAPPIST-1e      0.471175
Name: Transported, dtype: float64


In [11]:
train['AgeGroup'] = pd.cut(train['Age'], bins=[0, 12, 18, 30, 50, 100], labels=['Child', 'Teen', 'YoungAdult', 'Adult', 'Senior'])
print(train.groupby('AgeGroup', observed=True)['Transported'].mean())
print()
print(train.groupby('VIP')['Transported'].mean())

AgeGroup
Child         0.668790
Teen          0.537299
YoungAdult    0.468190
Adult         0.479432
Senior        0.484396
Name: Transported, dtype: float64

VIP
False    0.506332
True     0.381910
Name: Transported, dtype: float64


In [12]:
train['Group'] = train['PassengerId'].str.split('_').str[0]
group_size = train['Group'].value_counts()
train['GroupSize'] = train['Group'].map(group_size)
print(train.groupby('GroupSize')['Transported'].mean())

GroupSize
1    0.452445
2    0.538050
3    0.593137
4    0.640777
5    0.592453
6    0.614943
7    0.541126
8    0.394231
Name: Transported, dtype: float64


In [13]:
group_transported_std = train.groupby('Group')['Transported'].mean()
print(group_transported_std.value_counts(bins=[-0.01, 0.01, 0.5, 0.99, 1.01]))

(-0.011, 0.01]    2868
(0.99, 1.01]      2552
(0.01, 0.5]        565
(0.5, 0.99]        232
Name: count, dtype: int64


In [14]:
spending_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

train['TotalSpend'] = train[spending_cols].sum(axis=1)
test['TotalSpend'] = test[spending_cols].sum(axis=1)

train.loc[train['CryoSleep'].isna() & (train['TotalSpend'] == 0), 'CryoSleep'] = True
train.loc[train['CryoSleep'].isna() & (train['TotalSpend'] > 0), 'CryoSleep'] = False

test.loc[test['CryoSleep'].isna() & (test['TotalSpend'] == 0), 'CryoSleep'] = True
test.loc[test['CryoSleep'].isna() & (test['TotalSpend'] > 0), 'CryoSleep'] = False

print("Remaining missing CryoSleep in train:", train['CryoSleep'].isna().sum())
print("Remaining missing CryoSleep in test:", test['CryoSleep'].isna().sum())

Remaining missing CryoSleep in train: 0
Remaining missing CryoSleep in test: 0


In [15]:
for col in spending_cols:
    train.loc[(train['CryoSleep'] == True) & (train[col].isna()), col] = 0
    test.loc[(test['CryoSleep'] == True) & (test[col].isna()), col] = 0

for col in spending_cols:
    median_val = train.loc[train['CryoSleep'] == False, col].median()
    train.loc[(train['CryoSleep'] == False) & (train[col].isna()), col] = median_val
    test.loc[(test['CryoSleep'] == False) & (test[col].isna()), col] = median_val

print(train[spending_cols].isna().sum())
print(test[spending_cols].isna().sum())

RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
dtype: int64
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
dtype: int64


In [16]:
train['TotalSpend'] = train[spending_cols].sum(axis=1)
test['TotalSpend'] = test[spending_cols].sum(axis=1)

In [17]:
for col in ['HomePlanet', 'Destination', 'Deck', 'Side', 'VIP']:
    mode_val = train[col].mode()[0]
    train[col] = train[col].fillna(mode_val)
    test[col] = test[col].fillna(mode_val)

age_median = train['Age'].median()
train['Age'] = train['Age'].fillna(age_median)
test['Age'] = test['Age'].fillna(age_median)

print(train.isnull().sum())

PassengerId       0
HomePlanet        0
CryoSleep         0
Cabin           199
Destination       0
Age               0
VIP               0
RoomService       0
FoodCourt         0
ShoppingMall      0
Spa               0
VRDeck            0
Name            200
Transported       0
Deck              0
CabinNum        199
Side              0
AgeGroup        357
Group             0
GroupSize         0
TotalSpend        0
dtype: int64


In [18]:
train = train.drop(columns=['Cabin', 'Name', 'AgeGroup', 'PassengerId', 'Group'])
test_passenger_ids = test['PassengerId']
test = test.drop(columns=['Cabin', 'Name', 'PassengerId'])

print(train.isnull().sum())
print(train.shape, test.shape)

HomePlanet        0
CryoSleep         0
Destination       0
Age               0
VIP               0
RoomService       0
FoodCourt         0
ShoppingMall      0
Spa               0
VRDeck            0
Transported       0
Deck              0
CabinNum        199
Side              0
GroupSize         0
TotalSpend        0
dtype: int64
(8693, 16) (4277, 14)


In [19]:
train['CabinNum'] = pd.to_numeric(train['CabinNum'], errors='coerce')
test['CabinNum'] = pd.to_numeric(test['CabinNum'], errors='coerce')

cabinnum_median = train['CabinNum'].median()
train['CabinNum'] = train['CabinNum'].fillna(cabinnum_median)
test['CabinNum'] = test['CabinNum'].fillna(cabinnum_median)

print(train.isnull().sum())
print(test.isnull().sum())

HomePlanet      0
CryoSleep       0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Transported     0
Deck            0
CabinNum        0
Side            0
GroupSize       0
TotalSpend      0
dtype: int64
HomePlanet      0
CryoSleep       0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Deck            0
CabinNum        0
Side            0
TotalSpend      0
dtype: int64


In [20]:
train['CryoSleep'] = train['CryoSleep'].astype(int)
test['CryoSleep'] = test['CryoSleep'].astype(int)

train['VIP'] = train['VIP'].astype(int)
test['VIP'] = test['VIP'].astype(int)

train['Side'] = train['Side'].map({'P': 0, 'S': 1})
test['Side'] = test['Side'].map({'P': 0, 'S': 1})

train = pd.get_dummies(train, columns=['HomePlanet', 'Destination', 'Deck'])
test = pd.get_dummies(test, columns=['HomePlanet', 'Destination', 'Deck'])

print(train.shape, test.shape)
train.head()

(8693, 27) (4277, 25)


,CryoSleep,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,CabinNum,...,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,Deck_A,Deck_B,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,Deck_T
0,0,39.0,0,0.0,0.0,0.0,0.0,0.0,False,0.0,...,False,True,False,True,False,False,False,False,False,False
1,0,24.0,0,109.0,9.0,25.0,549.0,44.0,True,0.0,...,False,True,False,False,False,False,False,True,False,False
2,0,58.0,1,43.0,3576.0,0.0,6715.0,49.0,False,0.0,...,False,True,True,False,False,False,False,False,False,False
3,0,33.0,0,0.0,1283.0,371.0,3329.0,193.0,False,0.0,...,False,True,True,False,False,False,False,False,False,False
4,0,16.0,0,303.0,70.0,151.0,565.0,2.0,True,1.0,...,False,True,False,False,False,False,False,True,False,False


In [21]:
print("Train shape:", train.shape)
print("Test shape:", test.shape)

train_cols = set(train.columns) - {'Transported'}
test_cols = set(test.columns)

print("In train but not test:", train_cols - test_cols)
print("In test but not train:", test_cols - train_cols)

Train shape: (8693, 27)
Test shape: (4277, 25)
In train but not test: {'GroupSize'}
In test but not train: set()


In [22]:
test['Group'] = test_passenger_ids.str.split('_').str[0]
test_group_size = test['Group'].value_counts()
test['GroupSize'] = test['Group'].map(test_group_size)
test = test.drop(columns=['Group'])

print("Train shape:", train.shape)
print("Test shape:", test.shape)

train_cols = set(train.columns) - {'Transported'}
test_cols = set(test.columns)
print("In train but not test:", train_cols - test_cols)
print("In test but not train:", test_cols - train_cols)

Train shape: (8693, 27)
Test shape: (4277, 26)
In train but not test: set()
In test but not train: set()


In [23]:
from sklearn.model_selection import train_test_split

X = train.drop(columns=['Transported'])
y = train['Transported']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("y_train distribution:\n", y_train.value_counts(normalize=True))
print("y_val distribution:\n", y_val.value_counts(normalize=True))

X_train shape: (6954, 26)
X_val shape: (1739, 26)
y_train distribution:
 Transported
True     0.503595
False    0.496405
Name: proportion, dtype: float64
y_val distribution:
 Transported
True     0.503738
False    0.496262
Name: proportion, dtype: float64


In [24]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train, y_train)

y_pred = log_reg.predict(X_val)
accuracy = accuracy_score(y_val, y_pred)

print("Logistic Regression Validation Accuracy:", accuracy)

Logistic Regression Validation Accuracy: 0.7906843013225991


d:\Payoda_ML\ML-Assignment\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [25]:
from sklearn.preprocessing import StandardScaler

numeric_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'TotalSpend', 'CabinNum', 'GroupSize']

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_val_scaled[numeric_cols] = scaler.transform(X_val[numeric_cols])

log_reg2 = LogisticRegression(max_iter=1000, random_state=42)
log_reg2.fit(X_train_scaled, y_train)

y_pred2 = log_reg2.predict(X_val_scaled)
accuracy2 = accuracy_score(y_val, y_pred2)

print("Logistic Regression (scaled) Validation Accuracy:", accuracy2)

Logistic Regression (scaled) Validation Accuracy: 0.7872340425531915


In [26]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

log_reg_cv_scores = cross_val_score(log_reg2, X_train_scaled, y_train, cv=cv, scoring='accuracy')

print("CV Accuracy scores:", log_reg_cv_scores)
print("CV Mean Accuracy:", log_reg_cv_scores.mean())
print("CV Std Accuracy:", log_reg_cv_scores.std())

CV Accuracy scores: [0.78864127 0.79295471 0.79439252 0.78360891 0.7942446 ]
CV Mean Accuracy: 0.7907684032500815
CV Std Accuracy: 0.0041420079294290715


In [27]:
from xgboost import XGBClassifier

xgb = XGBClassifier(random_state=42, eval_metric='logloss')

xgb_cv_scores = cross_val_score(xgb, X_train, y_train, cv=cv, scoring='accuracy')

print("XGBoost CV Accuracy scores:", xgb_cv_scores)
print("XGBoost CV Mean Accuracy:", xgb_cv_scores.mean())
print("XGBoost CV Std Accuracy:", xgb_cv_scores.std())

XGBoost CV Accuracy scores: [0.80805176 0.79942487 0.79583034 0.8023005  0.80359712]
XGBoost CV Mean Accuracy: 0.8018409197875345
XGBoost CV Std Accuracy: 0.004094737791536764


In [28]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, log_loss
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_val)
y_proba_xgb = xgb.predict_proba(X_val)[:, 1]

print("Validation Accuracy:", accuracy_score(y_val, y_pred_xgb))
print("Precision:", precision_score(y_val, y_pred_xgb))
print("Recall:", recall_score(y_val, y_pred_xgb))
print("F1 Score:", f1_score(y_val, y_pred_xgb))
print("ROC-AUC:", roc_auc_score(y_val, y_proba_xgb))
print("Log Loss:", log_loss(y_val, y_proba_xgb))

Validation Accuracy: 0.8205865439907993
Precision: 0.827906976744186
Recall: 0.8127853881278538
F1 Score: 0.8202764976958525
ROC-AUC: 0.906904606951433
Log Loss: 0.38854289054870605


In [29]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 4, 5, 6, 7],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0]
}

random_search = RandomizedSearchCV(
    XGBClassifier(random_state=42, eval_metric='logloss'),
    param_distributions=param_dist,
    n_iter=30,
    cv=cv,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print("Best params:", random_search.best_params_)
print("Best CV Accuracy:", random_search.best_score_)

Best params: {'subsample': 0.9, 'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.1, 'colsample_bytree': 1.0}
Best CV Accuracy: 0.8088873487838055


In [30]:
best_xgb = random_search.best_estimator_
best_xgb.fit(X, y)

test_predictions = best_xgb.predict(test)

submission = pd.DataFrame({
    'PassengerId': test_passenger_ids,
    'Transported': test_predictions.astype(bool)
})

submission.to_csv('submission.csv', index=False)
submission.head()

ValueError: feature_names mismatch: ['CryoSleep', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'CabinNum', 'Side', 'GroupSize', 'TotalSpend', 'HomePlanet_Earth', 'HomePlanet_Europa', 'HomePlanet_Mars', 'Destination_55 Cancri e', 'Destination_PSO J318.5-22', 'Destination_TRAPPIST-1e', 'Deck_A', 'Deck_B', 'Deck_C', 'Deck_D', 'Deck_E', 'Deck_F', 'Deck_G', 'Deck_T'] ['CryoSleep', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'CabinNum', 'Side', 'TotalSpend', 'HomePlanet_Earth', 'HomePlanet_Europa', 'HomePlanet_Mars', 'Destination_55 Cancri e', 'Destination_PSO J318.5-22', 'Destination_TRAPPIST-1e', 'Deck_A', 'Deck_B', 'Deck_C', 'Deck_D', 'Deck_E', 'Deck_F', 'Deck_G', 'Deck_T', 'GroupSize']

In [31]:
test = test[X.columns]

test_predictions = best_xgb.predict(test)

submission = pd.DataFrame({
    'PassengerId': test_passenger_ids,
    'Transported': test_predictions.astype(bool)
})

submission.to_csv('submission.csv', index=False)
submission.head()

,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True


In [32]:
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': best_xgb.feature_importances_
}).sort_values('Importance', ascending=False)

print(importance_df.to_string(index=False))

                  Feature  Importance
                CryoSleep    0.236880
         HomePlanet_Earth    0.147729
               TotalSpend    0.109721
                   Deck_E    0.048705
        HomePlanet_Europa    0.045502
                      Spa    0.032392
                   Deck_G    0.032230
             ShoppingMall    0.031885
                FoodCourt    0.031123
                   VRDeck    0.030151
                     Side    0.029080
              RoomService    0.028455
                   Deck_B    0.026266
                   Deck_C    0.022899
Destination_PSO J318.5-22    0.020160
  Destination_55 Cancri e    0.019330
                 CabinNum    0.018292
                   Deck_F    0.015489
  Destination_TRAPPIST-1e    0.014991
          HomePlanet_Mars    0.014557
                      Age    0.011589
                GroupSize    0.009697
                   Deck_D    0.008133
                      VIP    0.007534
                   Deck_A    0.007213
            